In [1]:
import requests
import time
import pandas as pd
import csv
import re
import os
from datetime import datetime
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from dateutil.relativedelta import relativedelta  #is calender aware, as months are fixed No. days
from newsapi import NewsApiClient
from dotenv import load_dotenv


load_dotenv()
guardian_api_token = os.getenv('guardian_TOKEN') 
newsapi_token = os.getenv('newsapi_TOKEN')
newsapi = NewsApiClient(api_key=newsapi_token)

c:\Users\hmaxw\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_content_from_guardian(searches,start_date,end_date,api_key):
    content = []
    #is it 50 articels per page, so we go through mutliple pages that match with the same date needed
    for i in searches:
        time.sleep(5)  #worked before, but sometimes got connection blocked, this seems to have fixed it, but takes 10min to run
        page = 1                             #You may remove it and it may work, API seems rather temprimental
        while True:
            query = requests.get('https://content.guardianapis.com/search',params={'q' :i,'from-date':start_date,'to-date': end_date,'api-key': api_key,'page-size'  : 50,'page'       : page,
                                                                                    'show-fields': 'headline,body'}
                                                                                    )
            data = query.json()['response']
            content.extend(data['results'])
            if page >= data['pages']:
                break
            page += 1
    return content
#fields is a dict of title and content


searches = [
    '"Iran war" AND oil',
    '"Iran war" AND energy',
    '"Iran war" AND markets',
    '"Iran war" AND prices',
    '"oil price" AND Iran',
    'Strait of Hormuz',
    'Iran oil June 2025',
    'crude oil',
    '"crude oil" AND Iran AND war',
    '"energy market" AND Iran AND war',
    'energy market',
    'commodity market AND war',
    '"oil supply" AND Iran AND war',
    '"investor" AND Iran AND oil',
    '"stock market" AND energy',
    'stocks',
    'stock market',
]

raw_guardian_df = pd.DataFrame(get_content_from_guardian(searches, '2026-02-01',str(datetime.today().date()),guardian_api_token))

In [3]:
guardian_df = raw_guardian_df.copy()

In [4]:
guardian_df['date'] = pd.to_datetime(guardian_df['webPublicationDate']).dt.strftime('%Y%m%d')
guardian_df.drop_duplicates(subset=['webUrl'], inplace=True)
guardian_df = guardian_df[['date','fields']]
guardian_df.reset_index(drop=True, inplace=True)

In [5]:
guardian_df['headline'] = None
guardian_df['body_text'] = None

for i, item in enumerate(guardian_df['fields']):
    guardian_df.at[i, 'headline'] = item['headline']
    soup = BeautifulSoup(item['body'])
    body_text = soup.get_text()
    guardian_df.at[i, 'body_text'] = body_text

In [6]:
def get_date_pairs():
    current_date = datetime.today().date()
    start_date = current_date - relativedelta(months=1)

    date_pairs = []

    while start_date<current_date:
        next_day_date = start_date + relativedelta(days=1)
        date_pairs.append((start_date,next_day_date))
        start_date = next_day_date

    return date_pairs

In [7]:
def get_newsAPI_data():
    news_data = []
    for i in get_date_pairs():
        all_top_headlines = newsapi.get_everything(q='(oil OR crude OR energy OR brent OR wti) AND (iran OR price OR prices OR supply OR Stocks OR Energy market or stock market or commodity market OR demand OR production OR disruption OR volatility OR surge OR drop)',
                                           from_param=i[0],
                                           to=i[1],
                                           language='en',
                                           sort_by='relevancy')
        articles = all_top_headlines['articles']
        news_data.extend(articles)
    return news_data
    
news_data = get_newsAPI_data()

In [8]:
normalised_dataframe = pd.DataFrame(pd.json_normalize(news_data))
normalised_dataframe.drop_duplicates(subset=['title'],inplace=True)

if os.path.exists('NewsAPI_data_multipleday.csv'):
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, mode='a', header=False, encoding='utf-8')
    print('Dataset updated')

else:
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, encoding='utf-8')
    print('File does not exist, creating new')

Dataset updated


In [9]:
NewsAPI_to_get_relevant = pd.read_csv('NewsAPI_data_multipleday.csv',encoding='utf-8')
NewsAPI_to_get_relevant.drop_duplicates(subset='title', inplace=True)
NewsAPI_to_get_relevant['date'] = pd.to_datetime(NewsAPI_to_get_relevant['publishedAt']).dt.strftime('%Y%m%d')


In [10]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode('oil price energy markets Iran war investor sentiment volatility crude supply',convert_to_tensor=True)

def relevancy(headline):
    temp = model.encode(headline, convert_to_tensor=True)
    return util.cos_sim(temp,embeddings).item()

guardian_df['relevancy'] = guardian_df['headline'].apply(relevancy)
guardian_df = guardian_df[guardian_df['relevancy'] > 0.38]
NewsAPI_to_get_relevant['relevancy'] = NewsAPI_to_get_relevant['title'].apply(relevancy)
NewsAPI_to_get_relevant = NewsAPI_to_get_relevant[NewsAPI_to_get_relevant['relevancy'] > 0.38]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3118.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
guardian_df = guardian_df[['date','headline','body_text']]
guardian_df.reset_index(drop=True, inplace=True)
guardian_df.to_csv('guardian_data.csv', index=False, encoding='utf-8')

NewsAPI_to_get_relevant = NewsAPI_to_get_relevant[['date','title','content']]
NewsAPI_to_get_relevant.reset_index(drop=True, inplace=True)
NewsAPI_to_get_relevant.to_csv('NewsAPI_data.csv', index=False, encoding='utf-8')